# <center><b>Model training, selection, and insight collection</b></center>

In [2]:
%matplotlib qt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib

In [3]:
df = pd.read_csv('Final Dataset.csv')

In [4]:
# Changing columns names for more readability
df.columns = ['village_code', 'visit_month', 'spray_status', 'total_residents',
       'clinic_tests', 'total_rdts', 'avg_distance_to_health_facility_km',
       'incidence_per_1000', 'positivity_rate', 'dist_missing', 'rainfall',
       'rainfall_total', 'humidity', 'temp', 'max_temp', 'min_temp', 'temp_range',
       'prev_rainfall', 'prev2_rainfall', 'prev3_rainfall',
       'prev4_rainfall', 'prev_rainfall_total', 'prev2_rainfall_total',
       'prev3_rainfall_total', 'prev4_rainfall_total', 'prev_humidity',
       'prev2_humidity', 'prev3_humidity', 'prev4_humidity', 'prev_temp', 'prev2_temp',
       'prev3_temp', 'prev4_temp', 'prev_max_temp', 'prev2_max_temp',
       'prev3_max_temp', 'prev4_max_temp', 'prev_min_temp', 'prev2_min_temp',
       'prev3_min_temp', 'prev4_min_temp', 'prev_temp_range', 'prev2_temp_range',
       'prev3_temp_range', 'prev4_temp_range', 'prev_incidence_per_1000', 'mpi']

# Performing manual train-test split

In [5]:
village_codes = df['village_code'].unique()
village_codes.sort()
split_point_village = int(len(village_codes) * .8)
village_bool = df['village_code'] <= village_codes[split_point_village]
len(df[village_bool]), len(df[~village_bool])

(3019, 707)

In [6]:
dates = df['visit_month'].unique()
dates.sort()
split_point_date = int(len('visit_month') * .6)
month_bool = df['visit_month'] >= dates[split_point_date]
# I made these changes from village_bool because here the mean > median
len(df[~month_bool]), len(df[month_bool])
dates[split_point_date]

'2017-07-01'

In [7]:
def chrono(chrono_bool):
    """
    Function for specifying which boolean to use for the (chronological or spatial) split
    """
    return month_bool if chrono_bool else village_bool

In [8]:
?chrono

Signature: chrono(chrono_bool)
Docstring: Function for specifying which boolean to use for the (chronological or spatial) split
File:      c:\users\user\appdata\local\temp\ipykernel_11128\2877904609.py
Type:      function

In [9]:
# The model seems to do a worse job when performing a chronological split instead of a spatial split
# This is possibly because the train data has noisy data the model is forced to erraneously fit to and apply patterns on novel data.

In [10]:
# 80% / 20% chronological split
target_columns = ['incidence_per_1000'] # These are the columns our model tries to predict
# And these are poor or useless columns that carry information about which villages have suffered epidemics before
# Causing it to overfit by predicting historical average
useless_columns = ['total_residents', 'visit_month', 'village_code', 'avg_distance_to_health_facility_km', 'clinic_tests',
                   'rainfall', 'rainfall_total', 'total_rdts', 'positivity_rate'] \
                + ['prev_rainfall', 'prev_rainfall_total', 'prev_min_temp', 'prev_max_temp', 'prev_temp', 'prev_humidity'] \
                + ['prev2_rainfall_total', 'prev3_rainfall_total', 'prev4_rainfall_total', 'prev_temp_range', 'prev2_temp_range',\
                   'prev3_temp_range', 'prev4_temp_range', 'temp_range', 'mpi']
                 # + ['humidity', 'temp', 'max_temp', 'min_temp', 'temp_range', 'total_rdts']
# Train data becomes the majority (80%)
X_train = df[chrono(1)]
y_train = X_train[target_columns]
# Test data becomes the minority (20%)
X_test = df[~chrono(1)]
y_test = X_test[target_columns]
# We finally drop the target columns from the X data
X_train = X_train.drop(columns=target_columns + useless_columns) # We also don't need 'visit_month' anymore it isn't important in training
X_test = X_test.drop(columns=target_columns + useless_columns)

In [11]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
linear = LinearRegression().fit(X_train_scaled, y_train)

In [13]:
model = RandomForestRegressor(max_depth=10, random_state=42).fit(X_train_scaled, np.ravel(y_train))

In [14]:
dt = DecisionTreeRegressor().fit(X_train_scaled, y_train)

In [15]:
knn = KNeighborsRegressor().fit(X_train_scaled, y_train)

In [ ]:
ridge = GridSearchCV(
    Ridge(max_iter=2000),
    param_grid={ 'alpha': np.linspace(100, 1000, 10)},
    cv=5,
    n_jobs=2,
)
ridge.fit(X_train_scaled, y_train)

In [ ]:
lasso = GridSearchCV(
    Lasso(max_iter=2000),
    param_grid={ 'alpha': [.1, .2, .3, .4, .5] },
    cv=5,
    n_jobs=2,
)
lasso.fit(X_train_scaled, y_train)

In [ ]:
elasticnet = GridSearchCV(
    ElasticNet(max_iter=2000),
    param_grid={ 'alpha': [.001, .01, .1, 1, 100, 1000], 'l1_ratio': [.1, .3, .5, .7, .9] },
    cv=5,
    n_jobs=2,
)
elasticnet.fit(X_train_scaled, y_train)

In [ ]:
bv1_pred = []
for x in range(1, len(df['incidence_per_1000']) + 1):
    bv1_pred.append(df['incidence_per_1000'].mean())
bv1_pred = np.array(bv1_pred)

In [ ]:
metrics_bv1 = [100 * r2_score(bv1_pred, df['incidence_per_1000']),\
               mean_absolute_error(bv1_pred, df['incidence_per_1000']), \
               root_mean_squared_error(bv1_pred, df['incidence_per_1000'])]
metrics_bv1

In [ ]:
metrics_bv2 = knn_metrics = [100 * r2_score(df['prev_incidence_per_1000'], df['incidence_per_1000']),\
                             mean_absolute_error(df['prev_incidence_per_1000'], df['incidence_per_1000']), \
                             root_mean_squared_error(df['prev_incidence_per_1000'], df['incidence_per_1000'])]
metrics_bv2

### Wow! Look at the scores 😲

In [ ]:
# Note that this is a way less stable and, somewhat, less reliable of a model because it loses all of its predictive power on a chonological split.
knn_metrics = [100 * r2_score(y_test, knn.predict(X_test_scaled)), mean_absolute_error(y_test, knn.predict(X_test_scaled)), \
           root_mean_squared_error(y_test, knn.predict(X_test_scaled))]
knn_metrics

In [ ]:
# Not that this is a way less stable and, somewhat, less reliable of a model because it loses all of its predictive power on a chonological split.
linear_metrics = [100 * r2_score(y_test, linear.predict(X_test_scaled)), mean_absolute_error(y_test, linear.predict(X_test_scaled)), \
           root_mean_squared_error(y_test, linear.predict(X_test_scaled))]
linear_metrics

In [ ]:
metrics_dt = [100 * r2_score(y_test, dt.predict(X_test_scaled)), mean_absolute_error(y_test, dt.predict(X_test_scaled)), \
             root_mean_squared_error(y_test, dt.predict(X_test_scaled))]
metrics_dt

In [ ]:
metrics_ridge = [100 * r2_score(y_test, ridge.predict(X_test_scaled)), mean_absolute_error(y_test, ridge.predict(X_test_scaled)), \
                 root_mean_squared_error(y_test, ridge.predict(X_test_scaled))]
metrics_ridge

In [21]:
# This one is way more reliable
rf_metrics = [100 * r2_score(y_test, model.predict(X_test_scaled)), mean_absolute_error(y_test, model.predict(X_test_scaled)), \
          root_mean_squared_error(y_test, model.predict(X_test_scaled))]
rf_metrics

[62.86184371470165, 43.257901443028466, 74.94399721755238]

In [24]:
metrics_lasso = [100 * r2_score(y_test, lasso.predict(X_test_scaled)), mean_absolute_error(y_test, lasso.predict(X_test_scaled)), \
                 root_mean_squared_error(y_test, lasso.predict(X_test_scaled))]
metrics_lasso

[62.73715766047234, 40.931811842355486, 75.06969868192043]

In [25]:
metrics_enet = [100 * r2_score(y_test, elasticnet.predict(X_test_scaled)), mean_absolute_error(y_test, elasticnet.predict(X_test_scaled)), \
                root_mean_squared_error(y_test, elasticnet.predict(X_test_scaled))]
metrics_enet

[61.881240167045384, 41.28943568932236, 75.92696923343216]

In [26]:
# Sample data
data = [
    knn_metrics,
    rf_metrics,
    metrics_dt,
    linear_metrics,
    metrics_ridge,
    metrics_lasso,
    metrics_enet
]

# New labels for the groups (models)
group_labels = ['k-Nearest Neighbors', 'Random Forest', 'Decision Trees', 'Linear Regression', 'Ridge Regression', 'Lasso Regression', 'ElasticNet']
# New labels for the elements (metrics)
element_labels = ['R2 * 100 (0 - 100)', 'MAE', 'RMSE']
# Set the width of the bars
bar_width = 0.25
# Set the positions of the bar groups on the x-axis
x_positions = np.arange(len(group_labels))
# Create the plot
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the bars for each of the three elements in all groups
for i in range(3):
    values = [array[i] for array in data]
    ax.bar(x_positions + i * bar_width, values, bar_width, label=element_labels[i])

# Set the title and labels
ax.set_title('Model Performance Metrics Comparison')
ax.set_xlabel('Models')
ax.set_ylabel('Metric Value')

# Set the x-axis tick positions and labels
ax.set_xticks(x_positions + bar_width)
ax.set_xticklabels(group_labels, rotation=45, ha='right', fontsize=7)
# Add a legend to distinguish the elements
ax.legend()

plt.show()

### Let's create a baseline to predict our model against (definitely not for flexing 😏)

In [27]:
baseline = np.full(y_test.shape, y_test['incidence_per_1000'].mean())
mean_absolute_error(y_test, baseline)
# This baseline's accuracy is misleading. It's not that bad for a static prediction. It's because of the data's skewed structure

80.16705782559586

In [28]:
coef_df = pd.DataFrame({
    'features': X_train.columns,
    'coefs': ridge.best_estimator_.coef_
})
coef_df.sort_values('coefs')
plt.figure(figsize=(10, 6))
plt.xlabel('Features'); plt.ylabel('Importances')
plt.bar(coef_df['features'], coef_df['coefs'])
plt.xticks(rotation=45, ha='right', fontsize=7)

([0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21],
 [Text(0, 0, 'spray_status'),
  Text(1, 0, 'dist_missing'),
  Text(2, 0, 'humidity'),
  Text(3, 0, 'temp'),
  Text(4, 0, 'max_temp'),
  Text(5, 0, 'min_temp'),
  Text(6, 0, 'prev2_rainfall'),
  Text(7, 0, 'prev3_rainfall'),
  Text(8, 0, 'prev4_rainfall'),
  Text(9, 0, 'prev2_humidity'),
  Text(10, 0, 'prev3_humidity'),
  Text(11, 0, 'prev4_humidity'),
  Text(12, 0, 'prev2_temp'),
  Text(13, 0, 'prev3_temp'),
  Text(14, 0, 'prev4_temp'),
  Text(15, 0, 'prev2_max_temp'),
  Text(16, 0, 'prev3_max_temp'),
  Text(17, 0, 'prev4_max_temp'),
  Text(18, 0, 'prev2_min_temp'),
  Text(19, 0, 'prev3_min_temp'),
  Text(20, 0, 'prev4_min_temp'),
  Text(21, 0, 'prev_incidence_per_1000')])

##### This feels like cheating because the model is performing so well when prev_month_incidence is given as a feature
##### It's >3x more important than the best of the rest
##### But this is forecasting after all. So I guess it's fine

In [29]:
# Let's plot the feature importance and HOPE it's not overfitting
importances = model.feature_importances_
plt.figure(figsize=(20, 8))
plt.bar(X_test.columns, importances)
plt.xlabel('Features'); plt.ylabel('Importances')
plt.title('Feature Importance Bar Chart')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

In [30]:
corr_matrix = X_train.corr()

plt.figure(figsize=(20,8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.title("Feature Correlation Matrix")
plt.show()
# It's seems that rainfall and rainfall_total have a substantial similarity that dropping one won't hurt the model.
# Maybe something interesting to write on the paper

In [31]:
plt.scatter(y_test, lasso.predict(X_test_scaled), s=4)
plt.plot([0, 1000], [0, 1000], c='r')
plt.show()

#### The target variable has 60% unique values, that means the can't model has less chance of overfitting. 👍

In [32]:
len(df['prev_incidence_per_1000']), len(df['prev_incidence_per_1000'].unique())

(3726, 2217)

##### I should come to a conclusion in the paper that models predict better when given other quantifiable parameters that weren't listed here.
##### Climate data and spatial data alone isn't enough to reasonably predict malaria incidence in villages
##### People should focus on recording other possible parameters for better predictions
##### Because prev_incidence_per_1000 alone increased the R2 substantially, meaning it carries information other features don't

# <center>--------------------------------------------------------------------------------------------</center>

# Brainspace

##### The fact that village_code, total_rdts, and dist_missing make better predictors than climate data suggests something important

This is because they all have one thing in common: spatial specificity <br>
That should be why our model was predicting very poorly (sometimes even worse than our baseline), even with the best train / test split,
after removing parameters related to specific villages.
This suggests that it would be logical to add, in training and predictions, a parameter for showing how prone a village is to malaria incidences.<br>
<b>Note: We shouldn't just include village "codes" or IDs for each village. That would cause overfitting. It would just learn a village's historical average.</b><br>
Therefore, engineering a quantitative index (not a qualitative / one-hot encoded one, such as village codes) that would tell the model how prone the place is to malaria can boost model accuracy and interpretability.<br>
The next step is figuring out how to create such indexes. Maybe from previous incidences or other undocumented parameters.

In [35]:
plt.scatter(np.log1p(y_test), np.log1p(model.predict(X_test_scaled)), s=4); plt.plot([0, np.max(np.log1p(y_test))], [0, np.max(np.log1p(y_test))], c='r')

In [53]:
joblib.dump(knn, 'knn.pkl')
joblib.dump(model, 'rf.pkl')
joblib.dump(dt, 'dt.pkl')
joblib.dump(linear, 'linear.pkl')
joblib.dump(ridge, 'ridge.pkl')
joblib.dump(lasso, 'lasso.pkl')
joblib.dump(elasticnet, 'elasticnet.pkl')

['elasticnet.pkl']